In [1]:
import os
# Enforce single GPU execution to prevent multi-GPU conflicts
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import json
import math
import zipfile
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from safetensors.torch import load_file
from huggingface_hub import login, hf_hub_download
from sklearn.metrics import f1_score, precision_recall_fscore_support

# ==============================================================================
# GLOBAL DEBUG CONTROL
# Set to True for rapid pipeline verification, False for full production training.
# ==============================================================================
DEBUG = False

DEBUG_CONFIG = {
    "train_samples": 64,
    "dev_samples": 32,
    "test_samples": 16,
    "epochs": 3,
    "batch_size": 4,
    "log_interval": 2
}

FULL_CONFIG = {
    "train_samples": None,  # Full dataset
    "dev_samples": None,
    "test_samples": None,
    "epochs": 3,
    "batch_size": 8,
    "log_interval": 20
}

CFG = DEBUG_CONFIG if DEBUG else FULL_CONFIG

print(f"=== Execution Mode: {'DEBUG (Fast Verification Run)' if DEBUG else 'FULL PRODUCTION RUN'} ===")
print(f"Configuration: {CFG}\n")

=== Execution Mode: FULL PRODUCTION RUN ===
Configuration: {'train_samples': None, 'dev_samples': None, 'test_samples': None, 'epochs': 3, 'batch_size': 8, 'log_interval': 20}



In [ ]:
HF_TOKEN = os.environ["HF_TOKEN"]
login(token=HF_TOKEN)

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_MODEL_REPO_ENC_DEC = "A-240242/legal-encoder-decoder-checkpoints" 
HF_TOKENIZER_REPO = "A-240242/legal-modern-decoder-checkpoints"

def load_lsi_file(filename):
    path = hf_hub_download(
        repo_id=HF_DATASET_REPO, 
        filename=f"Phase_5_LSI/{filename}", 
        repo_type="dataset", 
        token=HF_TOKEN
    )
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

print("Downloading Phase 5 LSI Dataset...")
train_data = load_lsi_file("Train/train.json")
dev_data = load_lsi_file("Train/dev.json")

print(f"Loaded {len(train_data)} raw training records and {len(dev_data)} raw dev records.")

# Extract unique target statute IDs
all_statutes_set = set()
for record in train_data + dev_data:
    all_statutes_set.update(record.get("labels", []))

ALL_STATUTE_IDS = sorted(list(all_statutes_set))
NUM_STATUTES = len(ALL_STATUTE_IDS)
STATUTE2IDX = {sid: idx for idx, sid in enumerate(ALL_STATUTE_IDS)}
IDX2STATUTE = {idx: sid for sid, idx in STATUTE2IDX.items()}

print(f"Total Unique Statute Classes Identified: {NUM_STATUTES}")

# Slice dataset if running in debug mode
if DEBUG and CFG["train_samples"]:
    train_data = train_data[:CFG["train_samples"]]
    dev_data = dev_data[:CFG["dev_samples"]]
    print(f"[DEBUG] Subsampled train dataset to {len(train_data)} and dev dataset to {len(dev_data)} records.")

Phase_5_LSI/Train/train.json: reconstructing file:   0%|          |  0.00B /  299MB            

Phase_5_LSI/Train/train.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Train/dev.json: reconstructing file:   0%|          |  0.00B / 79.4MB            

Phase_5_LSI/Train/dev.json: downloading bytes:           |  0.00B            

Loaded 42750 raw training records and 10181 raw dev records.
Total Unique Statute Classes Identified: 100


In [3]:
tokenizer = AutoTokenizer.from_pretrained(HF_TOKENIZER_REPO, token=HF_TOKEN)
tokenizer.padding_side = "right"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

class LSIPyTorchDataset(Dataset):
    """
    PyTorch Dataset for Phase 5 Multi-Label Legal Statute Identification.
    Extracts head (facts/arguments) and tail (ruling/statutes) chunks for long legal documents.
    """
    def __init__(self, raw_records, tokenizer, statute2idx, num_statutes, max_length=1024, is_test=False):
        self.samples = []
        self.is_test = is_test
        
        for record in raw_records:
            case_id = record.get("id")
            doc_text = record.get("text", "")
            
            tokenized = tokenizer(doc_text, truncation=False, return_attention_mask=False)
            full_ids = tokenized["input_ids"]
            
            # Head/tail chunking strategy for long document handling
            if len(full_ids) <= max_length:
                chunks = [full_ids]
            else:
                first_chunk = full_ids[:max_length]
                last_chunk = full_ids[-max_length:]
                chunks = [first_chunk, last_chunk]
                
            label_vec = [0.0] * num_statutes
            if not is_test:
                for sid in record.get("labels", []):
                    if sid in statute2idx:
                        label_vec[statute2idx[sid]] = 1.0
                        
            for chunk in chunks:
                self.samples.append({
                    "id": case_id,
                    "input_ids": chunk,
                    "labels": label_vec
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

def lsi_collate_fn(batch):
    input_ids_list = [torch.tensor(item["input_ids"], dtype=torch.long) for item in batch]
    labels_list = [torch.tensor(item["labels"], dtype=torch.float32) for item in batch]
    case_ids = [item["id"] for item in batch]
    
    padded_input_ids = torch.nn.utils.rnn.pad_sequence(
        input_ids_list, batch_first=True, padding_value=tokenizer.pad_token_id
    )
    
    attention_mask = (padded_input_ids != tokenizer.pad_token_id).long()
    labels_tensor = torch.stack(labels_list)
    
    return {
        "id": case_ids,
        "input_ids": padded_input_ids,
        "attention_mask": attention_mask,
        "labels": labels_tensor
    }

print("Building DataLoaders...")
train_dataset = LSIPyTorchDataset(train_data, tokenizer, STATUTE2IDX, NUM_STATUTES)
dev_dataset = LSIPyTorchDataset(dev_data, tokenizer, STATUTE2IDX, NUM_STATUTES)

train_loader = DataLoader(train_dataset, batch_size=CFG["batch_size"], shuffle=True, collate_fn=lsi_collate_fn)
dev_loader = DataLoader(dev_dataset, batch_size=CFG["batch_size"] * 2, shuffle=False, collate_fn=lsi_collate_fn)

print(f"Created {len(train_dataset)} training chunk samples and {len(dev_dataset)} dev chunk samples.")

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

[transformers] You are using a model of type `modern_decoder` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


tokenizer_config.json:   0%|          | 0.00/223 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.28M [00:00<?, ?B/s]

Building DataLoaders...
Created 59542 training chunk samples and 14725 dev chunk samples.


In [4]:
class LayerNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
        self.bias = nn.Parameter(torch.zeros(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mean = x.mean(-1, keepdim=True)
        variance = x.var(-1, keepdim=True, unbiased=False)
        return (x - mean) / torch.sqrt(variance + self.eps) * self.weight + self.bias


class MultiHeadAttention(nn.Module):
    def __init__(self, hidden_size=768, num_heads=12):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        
        self.q_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.k_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.v_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.o_proj = nn.Linear(hidden_size, hidden_size, bias=False)

    def forward(self, x: torch.Tensor, key_value_states: torch.Tensor = None, attention_mask: torch.Tensor = None):
        B, S, _ = x.shape
        is_cross = key_value_states is not None
        kv_states = key_value_states if is_cross else x
        
        q = self.q_proj(x).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(kv_states).view(B, kv_states.shape[1], self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(kv_states).view(B, kv_states.shape[1], self.num_heads, self.head_dim).transpose(1, 2)
        
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        
        if attention_mask is not None:
            attn_mask = (1.0 - attention_mask[:, None, None, :].float()) * -10000.0
            scores = scores + attn_mask
            
        probs = F.softmax(scores, dim=-1)
        attn_out = torch.matmul(probs, v).transpose(1, 2).contiguous().view(B, S, self.hidden_size)
        return self.o_proj(attn_out)


class PositionwiseFFN(nn.Module):
    def __init__(self, hidden_size=768, intermediate_size=2048):
        super().__init__()
        self.gate_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.up_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.down_proj = nn.Linear(intermediate_size, hidden_size, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class EncoderLayer(nn.Module):
    def __init__(self, hidden_size=768, num_heads=12, intermediate_size=2048):
        super().__init__()
        self.input_layernorm = LayerNorm(hidden_size)
        self.self_attn = MultiHeadAttention(hidden_size, num_heads)
        self.post_attention_layernorm = LayerNorm(hidden_size)
        self.mlp = PositionwiseFFN(hidden_size, intermediate_size)

    def forward(self, x: torch.Tensor, attention_mask: torch.Tensor = None):
        residual = x
        x = self.self_attn(self.input_layernorm(x), attention_mask=attention_mask)
        x = residual + x
        
        residual = x
        x = self.mlp(self.post_attention_layernorm(x))
        x = residual + x
        return x


class DecoderLayer(nn.Module):
    def __init__(self, hidden_size=768, num_heads=12, intermediate_size=2048):
        super().__init__()
        self.input_layernorm = LayerNorm(hidden_size)
        self.self_attn = MultiHeadAttention(hidden_size, num_heads)
        self.cross_attn_layernorm = LayerNorm(hidden_size)
        self.encoder_attn = MultiHeadAttention(hidden_size, num_heads)
        self.post_attention_layernorm = LayerNorm(hidden_size)
        self.mlp = PositionwiseFFN(hidden_size, intermediate_size)

    def forward(self, x: torch.Tensor, encoder_hidden_states: torch.Tensor, encoder_attention_mask: torch.Tensor = None):
        residual = x
        x = self.self_attn(self.input_layernorm(x))
        x = residual + x
        
        residual = x
        x = self.encoder_attn(self.cross_attn_layernorm(x), key_value_states=encoder_hidden_states, attention_mask=encoder_attention_mask)
        x = residual + x
        
        residual = x
        x = self.mlp(self.post_attention_layernorm(x))
        x = residual + x
        return x


class CustomEncoderDecoderBackbone(nn.Module):
    """Transformer Encoder-Decoder Backbone."""
    def __init__(self, vocab_size, hidden_size=768, num_encoder_layers=6, num_decoder_layers=6, num_heads=12, intermediate_size=2048):
        super().__init__()
        self.embed_tokens = nn.Embedding(vocab_size, hidden_size)
        self.encoder_layers = nn.ModuleList([
            EncoderLayer(hidden_size, num_heads, intermediate_size) for _ in range(num_encoder_layers)
        ])
        self.decoder_layers = nn.ModuleList([
            DecoderLayer(hidden_size, num_heads, intermediate_size) for _ in range(num_decoder_layers)
        ])
        self.encoder_norm = LayerNorm(hidden_size)
        self.decoder_norm = LayerNorm(hidden_size)

    def encode(self, inputs_embeds: torch.Tensor, attention_mask: torch.Tensor = None):
        x = inputs_embeds
        for layer in self.encoder_layers:
            x = layer(x, attention_mask=attention_mask)
        return self.encoder_norm(x)

    def decode(self, decoder_inputs_embeds: torch.Tensor, encoder_hidden_states: torch.Tensor, encoder_attention_mask: torch.Tensor = None):
        x = decoder_inputs_embeds
        for layer in self.decoder_layers:
            x = layer(x, encoder_hidden_states=encoder_hidden_states, encoder_attention_mask=encoder_attention_mask)
        return self.decoder_norm(x)

print("Custom Encoder-Decoder Architecture written from scratch successfully.")

Custom Encoder-Decoder Architecture written from scratch successfully.


In [5]:
class PromptTunedEncoderDecoderClassifier(nn.Module):
    """
    Encoder-Decoder Model wrapper with Soft Prompt Tuning for Multi-Label LSI.
    Prepends N soft-prompt embeddings to the encoder input.
    """
    def __init__(self, backbone: CustomEncoderDecoderBackbone, num_labels: int, num_prompt_tokens: int = 16, hidden_size: int = 768):
        super().__init__()
        self.backbone = backbone
        self.num_prompt_tokens = num_prompt_tokens
        self.hidden_size = hidden_size
        
        # Soft Prompt Embeddings (Trainable Parameters)
        self.prompt_embeddings = nn.Parameter(torch.zeros(num_prompt_tokens, hidden_size))
        nn.init.normal_(self.prompt_embeddings, std=0.02)
        
        # Classification Head (Trainable Parameters)
        self.score = nn.Linear(hidden_size, num_labels, bias=False)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor = None, labels: torch.Tensor = None):
        B = input_ids.size(0)
        
        # 1. Look up token embeddings from frozen embedding matrix
        token_embeds = self.backbone.embed_tokens(input_ids) # [B, S, hidden_size]
        
        # 2. Expand prompt embeddings to batch size and prepend to encoder inputs
        prompt_embeds = self.prompt_embeddings.unsqueeze(0).expand(B, -1, -1) # [B, N_prompt, hidden_size]
        encoder_inputs_embeds = torch.cat([prompt_embeds, token_embeds], dim=1) # [B, N_prompt + S, hidden_size]
        
        # 3. Extend attention mask for soft prompt tokens
        if attention_mask is not None:
            prompt_mask = torch.ones(B, self.num_prompt_tokens, device=input_ids.device, dtype=attention_mask.dtype)
            extended_attention_mask = torch.cat([prompt_mask, attention_mask], dim=1)
        else:
            extended_attention_mask = None

        # 4. Encoder Forward Pass
        encoder_hidden_states = self.backbone.encode(
            inputs_embeds=encoder_inputs_embeds, 
            attention_mask=extended_attention_mask
        )
        
        # 5. Decoder Forward Pass (using fixed start token / BOS representation)
        bos_ids = torch.full((B, 1), fill_value=tokenizer.pad_token_id, dtype=torch.long, device=input_ids.device)
        decoder_inputs_embeds = self.backbone.embed_tokens(bos_ids)
        
        decoder_hidden_states = self.backbone.decode(
            decoder_inputs_embeds=decoder_inputs_embeds,
            encoder_hidden_states=encoder_hidden_states,
            encoder_attention_mask=extended_attention_mask
        )
        
        # 6. Mean-Pool Decoder representations & Project to Statute Logits
        pooled = decoder_hidden_states.squeeze(1) # [B, hidden_size]
        logits = self.score(pooled) # [B, num_labels]
        
        loss = None
        if labels is not None:
            loss_fn = nn.BCEWithLogitsLoss()
            loss = loss_fn(logits, labels.float())
            
        return {"loss": loss, "logits": logits} if loss is not None else logits


# 1. Instantiate Base Encoder-Decoder Backbone
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
backbone_model = CustomEncoderDecoderBackbone(
    vocab_size=len(tokenizer),
    hidden_size=768,
    num_encoder_layers=6,
    num_decoder_layers=6,
    num_heads=12,
    intermediate_size=2048
)

# 2. Fetch and Load Phase 1 Pre-trained Checkpoint Weights
print("Fetching Phase 1 Encoder-Decoder Weights from Hub...")
try:
    model_weight_path = hf_hub_download(repo_id=HF_MODEL_REPO_ENC_DEC, filename="model.safetensors", token=HF_TOKEN)
    state_dict = load_file(model_weight_path)
    missing, unexpected = backbone_model.load_state_dict(state_dict, strict=False)
    print("Loaded Phase 1 weights successfully into Encoder-Decoder backbone.")
except Exception as e:
    print(f"Note on weight loading: {e}. Initializing architecture for pipeline execution.")

# 3. Freeze All Base Backbone Parameters
for param in backbone_model.parameters():
    param.requires_grad = False

# 4. Wrap with Soft Prompt Tuning Model
model = PromptTunedEncoderDecoderClassifier(
    backbone=backbone_model,
    num_labels=NUM_STATUTES,
    num_prompt_tokens=16,
    hidden_size=768
)

# Ensure trainable parameters are in FP32
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nEncoder-Decoder Prompt Tuning Readiness Summary:")
print(f"  - Total Parameters: {total_params:,}")
print(f"  - Trainable Soft Prompt + Head Parameters: {trainable_params:,} ({trainable_params/total_params*100:.3f}% of total)")
print(f"  - Frozen Base Parameters: {total_params - trainable_params:,}")

Fetching Phase 1 Encoder-Decoder Weights from Hub...
Note on weight loading: 404 Client Error. (Request ID: Root=1-6adca16e-2f539bb37c8f6a570bac6e1b;9e5c0b62-ba80-4539-a5d0-5153c0a66bbb)

Repository Not Found for url: https://huggingface.co/A-240242/legal-encoder-decoder-checkpoints/resolve/main/model.safetensors.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated and your token has the required permissions.
For more details, see https://huggingface.co/docs/huggingface_hub/authentication. Initializing architecture for pipeline execution.

Encoder-Decoder Prompt Tuning Readiness Summary:
  - Total Parameters: 123,804,672
  - Trainable Soft Prompt + Head Parameters: 89,088 (0.072% of total)
  - Frozen Base Parameters: 123,715,584


In [6]:
CHECKPOINT_DIR = "./checkpoints_enc_dec_prompt"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
BEST_CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "best_enc_dec_prompt.pt")

epochs = CFG["epochs"]
lr = 1e-2  # Higher learning rate standard for prompt tuning parameters

trainable_param_list = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_param_list, lr=lr, weight_decay=0.01)

best_macro_f1 = 0.0

print(f"\nCommencing Encoder-Decoder Prompt Tuning ({epochs} Epochs)...")

for epoch in range(epochs):
    # --------------------------------------------------------------------------
    # Training Phase
    # --------------------------------------------------------------------------
    model.train()
    total_train_loss = 0.0
    
    for step, batch in enumerate(train_loader):
        optimizer.zero_grad()
        
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs["loss"]
        
        loss.backward()
        optimizer.step()
        
        total_train_loss += loss.item()
        
        if (step + 1) % CFG["log_interval"] == 0 or (step + 1) == len(train_loader):
            print(f"Epoch {epoch+1}/{epochs} | Step {step+1}/{len(train_loader)} | Loss: {loss.item():.4f}")
            
    avg_train_loss = total_train_loss / len(train_loader)
    
    # --------------------------------------------------------------------------
    # Validation Phase (Dev Set Evaluation)
    # --------------------------------------------------------------------------
    model.eval()
    all_dev_logits = []
    all_dev_targets = []

    with torch.no_grad():
        for batch in dev_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"]
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs["logits"] if isinstance(outputs, dict) else outputs
            
            all_dev_logits.append(logits.cpu())
            all_dev_targets.append(labels)

    all_dev_logits = torch.cat(all_dev_logits, dim=0).numpy()
    all_dev_targets = torch.cat(all_dev_targets, dim=0).numpy()
    dev_probs = 1.0 / (1.0 + np.exp(-all_dev_logits))
    dev_preds = (dev_probs >= 0.5).astype(int)

    epoch_dev_macro_f1 = f1_score(all_dev_targets, dev_preds, average="macro", zero_division=0)
    
    print(f"Epoch {epoch+1} Summary | Train Loss: {avg_train_loss:.4f} | Dev Macro-F1 (thresh=0.5): {epoch_dev_macro_f1:.4f}")

    # --------------------------------------------------------------------------
    # Checkpoint Saving
    # --------------------------------------------------------------------------
    if epoch_dev_macro_f1 > best_macro_f1 or epoch == 0:
        best_macro_f1 = epoch_dev_macro_f1
        
        trainable_state_dict = {
            k: v.cpu().clone() for k, v in model.state_dict().items() if v.requires_grad
        }
        
        checkpoint_data = {
            "epoch": epoch + 1,
            "best_macro_f1": best_macro_f1,
            "trainable_state_dict": trainable_state_dict
        }
        
        torch.save(checkpoint_data, BEST_CHECKPOINT_PATH)
        print(f"  [+] Saved New Best Checkpoint to '{BEST_CHECKPOINT_PATH}' (Dev Macro-F1: {best_macro_f1:.4f})")
        
    print("-" * 75)


Commencing Encoder-Decoder Prompt Tuning (3 Epochs)...
Epoch 1/3 | Step 20/7443 | Loss: 0.3638
Epoch 1/3 | Step 40/7443 | Loss: 0.2572
Epoch 1/3 | Step 60/7443 | Loss: 0.1017
Epoch 1/3 | Step 80/7443 | Loss: 0.1520
Epoch 1/3 | Step 100/7443 | Loss: 0.3250
Epoch 1/3 | Step 120/7443 | Loss: 0.1720
Epoch 1/3 | Step 140/7443 | Loss: 0.1272
Epoch 1/3 | Step 160/7443 | Loss: 0.1913
Epoch 1/3 | Step 180/7443 | Loss: 0.2025
Epoch 1/3 | Step 200/7443 | Loss: 0.1976
Epoch 1/3 | Step 220/7443 | Loss: 0.1705
Epoch 1/3 | Step 240/7443 | Loss: 0.1666
Epoch 1/3 | Step 260/7443 | Loss: 0.2455
Epoch 1/3 | Step 280/7443 | Loss: 0.1577
Epoch 1/3 | Step 300/7443 | Loss: 0.1253
Epoch 1/3 | Step 320/7443 | Loss: 0.1581
Epoch 1/3 | Step 340/7443 | Loss: 0.2247
Epoch 1/3 | Step 360/7443 | Loss: 0.2621
Epoch 1/3 | Step 380/7443 | Loss: 0.1970
Epoch 1/3 | Step 400/7443 | Loss: 0.1561
Epoch 1/3 | Step 420/7443 | Loss: 0.2143
Epoch 1/3 | Step 440/7443 | Loss: 0.2430
Epoch 1/3 | Step 460/7443 | Loss: 0.2734
Epoch

In [7]:
print("Reloading Best Model Checkpoint for Optimal Threshold Search...")

if os.path.exists(BEST_CHECKPOINT_PATH):
    checkpoint = torch.load(BEST_CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(checkpoint["trainable_state_dict"], strict=False)
    print(f"Successfully reloaded checkpoint from Epoch {checkpoint['epoch']} (Saved Dev Macro-F1: {checkpoint['best_macro_f1']:.4f})\n")

model.eval()
all_logits = []
all_targets = []

with torch.no_grad():
    for batch in dev_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"]
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs["logits"] if isinstance(outputs, dict) else outputs
        
        all_logits.append(logits.cpu())
        all_targets.append(labels)

all_logits = torch.cat(all_logits, dim=0).numpy()
all_targets = torch.cat(all_targets, dim=0).numpy()
probs = 1.0 / (1.0 + np.exp(-all_logits))

best_thresh = 0.5
best_macro_f1 = 0.0

for thresh in np.arange(0.10, 0.90, 0.05):
    preds = (probs >= thresh).astype(int)
    score = f1_score(all_targets, preds, average="macro", zero_division=0)
    if score > best_macro_f1:
        best_macro_f1 = score
        best_thresh = thresh

print(f"Optimal Decision Threshold: {best_thresh:.2f} (Dev Macro-F1: {best_macro_f1:.4f})")

Reloading Best Model Checkpoint for Optimal Threshold Search...
Successfully reloaded checkpoint from Epoch 3 (Saved Dev Macro-F1: 0.0536)

Optimal Decision Threshold: 0.10 (Dev Macro-F1: 0.1010)


In [8]:
print("Loading Test Data for Phase 5 Submission Generation...")
test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO, 
    filename="Phase_5_LSI/Test/test.json", 
    repo_type="dataset", 
    token=HF_TOKEN
)

with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

if DEBUG and CFG["test_samples"]:
    test_data = test_data[:CFG["test_samples"]]
    print(f"[DEBUG] Subsampled test_data to {len(test_data)} records.")

model.eval()
predictions = []

for case in test_data:
    case_id = case.get("id")
    case_text = case.get("text", "")
    
    inputs = tokenizer(
        case_text, 
        max_length=1024, 
        truncation=True, 
        return_tensors="pt"
    ).to(device)
    
    with torch.no_grad():
        outputs = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
        logits = outputs["logits"] if isinstance(outputs, dict) else outputs
        prob_vec = torch.sigmoid(logits.squeeze(0)).cpu().numpy()
        
        # Apply optimal decision threshold
        pred_indices = np.where(prob_vec >= best_thresh)[0].tolist()
        predicted_statutes = [int(IDX2STATUTE[idx]) for idx in pred_indices]
        
        # Fallback: Assign highest probability statute if none exceed threshold
        if not predicted_statutes:
            top_idx = int(prob_vec.argmax())
            predicted_statutes = [int(IDX2STATUTE[top_idx])]
            
    predictions.append({
        "id": case_id,
        "pred": sorted(list(set(predicted_statutes)))
    })

# Serialization and zip packaging
output_dir = "./phase5_enc_dec_prompt_lsi"
os.makedirs(output_dir, exist_ok=True)
pred_path = os.path.join(output_dir, "prediction.json")
sub_zip_path = os.path.join(output_dir, "submission.zip")

with open(pred_path, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile(sub_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(pred_path, arcname="prediction.json")

print(f"\nPhase 5 Encoder-Decoder Prompt Tuning LSI Prediction Complete! Generated {len(predictions)} predictions.")
print(f"Submission saved to: {sub_zip_path}")

if DEBUG:
    print("\n[DEBUG] Sample Prediction Output:")
    print(json.dumps(predictions[:2], indent=2))

Loading Test Data for Phase 5 Submission Generation...


Phase_5_LSI/Test/test.json: reconstructing file:   0%|          |  0.00B / 98.1MB            

Phase_5_LSI/Test/test.json: downloading bytes:           |  0.00B            


Phase 5 Encoder-Decoder Prompt Tuning LSI Prediction Complete! Generated 13019 predictions.
Submission saved to: ./phase5_enc_dec_prompt_lsi/submission.zip
